### Load change records into the raw layer of the Air Travel warehouse

#### Don't run this notebook before you have:
* copied your raw tables to the inc_ dataset
* Ran the orchestrate_first.sh to create and populate the snapshot and model tables with the initial data.

#### SDK documentation links (in case you need to modify the common function):
*   [BQ Client](https://cloud.google.com/python/docs/reference/bigquery/latest/google.cloud.bigquery.client.Client)
*   [LoadJobConfig](https://cloud.google.com/python/docs/reference/bigquery/latest/google.cloud.bigquery.job.LoadJobConfig)


#### Common functions

In [11]:
from google.cloud import bigquery

project_id = "segfault-434120"
bucket = "skincare-products"
parent_folder = "incrementals"
region = "us-central1"
dataset = "inc_skincare_products_raw" # be sure to add the "in_" prefix

bq_client = bigquery.Client()

def load_table_from_csv(folder, file_name, table, schema, delimiter=",", quote_character="\"", skip_leading_rows=1):

  uri = f"gs://{bucket}/{parent_folder}/{folder}/{file_name}"
  table_id = f"{project_id}.{dataset}.{table}"

  job_config = bigquery.LoadJobConfig(
        schema=schema,
        skip_leading_rows=skip_leading_rows,
        source_format=bigquery.SourceFormat.CSV,
        create_disposition=bigquery.CreateDisposition.CREATE_NEVER,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
        field_delimiter=delimiter,
        quote_character=quote_character,
        allow_jagged_rows=True,
        allow_quoted_newlines=True,
        ignore_unknown_values=True
      )

  load_job = bq_client.load_table_from_uri(uri, table_id, job_config=job_config)
  load_job.result()

  destination_table = bq_client.get_table(table_id)
  print("Table has {} rows after load.".format(destination_table.num_rows))

def load_table_from_json(folder, file_name, table, schema):

  uri = f"gs://{bucket}/{parent_folder}/{folder}/{file_name}"
  table_id = f"{project_id}.{dataset}.{table}"

  job_config = bigquery.LoadJobConfig(schema=schema,
      source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
      create_disposition=bigquery.CreateDisposition.CREATE_NEVER,
      write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
  )

  load_job = bq_client.load_table_from_uri(
      uri,
      table_id,
      location=region,
      job_config=job_config,
  )

  load_job.result()

  destination_table = bq_client.get_table(table_id)
  print("Table has {} rows after load.".format(destination_table.num_rows))

#### Load `cosmetic_ingredient`

Removed the `_load_time` and `_data_source` fields from the schema below because we are not creating a new table, just loading into an existing one

In [6]:
folder = "cosmetic_ingredients"
file_name = "*.csv"
table = "cosmetic_ingredient"
delimiter = ","

schema = [
  bigquery.SchemaField("cosing_ref_no", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("ingredient_unique_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("ingredient_common_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("european_pharmacopoeia_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("cas_no", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("ec_no", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("chemical_description", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("restriction", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("function", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("update_date", "STRING", mode="NULLABLE"),
]

load_table_from_csv(folder, file_name, table, schema, delimiter, skip_leading_rows=9) # Skipping metadata rows

Table has 57428 rows after load.


#### Load `dermatologic_diseases`

Removed the `_load_time` and `_data_source` fields from the schema below because we are not creating a new table, just loading into an existing one

Remove leading and trailing whitespace from files

In [12]:
folder = "dermatologic_diseases/llm_text"
file_name = "*.txt"
table = "dermatologic_disease"

schema = [
  bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("physical_description", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("causes", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("treatments", "STRING", mode="NULLABLE")
]

load_table_from_json(folder, file_name, table, schema)

Table has 740 rows after load.


#### Create and load `sephora_product`

Remove leading and trailing whitespace from files

In [13]:
folder = "sephora_products"
file_name = "*.csv"
table = "sephora_product"
delimiter = ","

schema = [
  bigquery.SchemaField("product_id", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("product_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("brand_id", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("brand_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("loves_count", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("rating", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("reviews", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("size", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("variation_type", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("variation_value", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("variation_desc", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("ingredients", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("price_usd", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("value_price_usd", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("sale_price_usd", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("limited_edition", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("new", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("online_only", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("out_of_stock", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("sephora_exclusive", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("highlights", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("primary_category", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("secondary_category", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("tertiary_category", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("child_count", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("child_max_price", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("child_min_price", "FLOAT", mode="NULLABLE"),
]

load_table_from_csv(folder, file_name, table, schema, delimiter)

Table has 16988 rows after load.


#### Create and load `sephora_products_reviews`

In [16]:
folder = "sephora_products_reviews"
file_name = "*.csv"
table = "sephora_product_review"
delimiter = ","



schema = [
  bigquery.SchemaField("row_number", "INTEGER", mode="NULLABLE"), # Unlabled Row
  bigquery.SchemaField("author_id", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("rating", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("is_recommended", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("helpfulness", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("total_feedback_count", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("total_neg_feedback_count", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("total_pos_feedback_count", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("submission_time", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("review_text", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("review_title", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("skin_tone", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("eye_color", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("skin_type", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("hair_color", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("product_id", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("product_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("brand_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("price_usd", "FLOAT", mode="NULLABLE"),
]

load_table_from_csv(folder, file_name, table, schema, delimiter)

Table has 2188822 rows after load.
